# Raw Data Preprocessing

# 1. Imports

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from scipy.stats import f_oneway

# 2. Raw Data Import

In [ ]:
data_2023 = "../data/raw/SUAC_2023.csv"
data_2024 = "../data/raw/SUAC_2024.csv"
data_2025 = "../data/raw/SUAC_2025.csv"
data_2026 = "../data/raw/SUAC_2026.csv"

In [ ]:
df_2023 = pd.read_csv(data_2023)
df_2024 = pd.read_csv(data_2024)
df_2025 = pd.read_csv(data_2025)
df_2026 = pd.read_csv(data_2026)

# 3. Initial Exploration

In [ ]:
df_2023.head()

In [ ]:
df_2024.head()

In [ ]:
df_2025.head()

In [ ]:
df_2026.head()

## 3.1 Consistency Validation

In [ ]:
dfs = {
    2023: df_2023,
    2024: df_2024,
    2025: df_2025,
    2026: df_2026,
}

def explore_dfs(df, name=""):
    print("=" * 70)
    print(f"  INITIAL EXPLORATION — {name}")
    print("=" * 70)

    print(f"\nDimensions: {df.shape[0]} rows × {df.shape[1]} columns\n")

    print("--- info() ---")
    df.info()

In [ ]:
for year, df in dfs.items():
    explore_dfs(df, name=f"Year {year}")

## 3.2 DataFrame Concatenation

In [ ]:
list_df = [df_2023,df_2024,df_2025,df_2026]
for df in list_df:
    df['codigo_postal_solicitud'] = df['codigo_postal_solicitud'].astype(str)

In [ ]:
df_complete = pd.concat(list_df, ignore_index=True)

## 3.3 Final DataFrame Exploration

In [ ]:
df_complete.head()

In [ ]:
df_complete.shape

In [ ]:
df_complete.describe()

In [ ]:
df_complete.isnull().mean()

In [ ]:
df_complete.info()

In [ ]:
df_complete.nunique(dropna=True)

# 4. Data Cleaning

## 4.1 Drop Columns with +40% null

In [ ]:
df_complete.isnull().mean()*100

In [ ]:
null_columns = ['edad','colonia_solicitud','latitud','longitud']
df_complete.drop(labels=null_columns, axis=1,inplace=True)
df_complete

In [ ]:
df_complete.isnull().mean()*100

## 4.2 Date and Time Handling

### 4.2.1 Parse dates and times

In [ ]:
df_complete['fecha_solicitud'] = pd.to_datetime(df_complete['fecha_solicitud'], format='%Y-%m-%d')
df_complete

In [ ]:
df_complete['fecha_concluido'] = pd.to_datetime(df_complete['fecha_concluido'], format='%Y-%m-%d')
df_complete

In [ ]:
df_complete['hora_solicitud_dt'] = pd.to_datetime(df_complete['hora_solicitud'], format='mixed', errors='coerce')
df_complete

In [ ]:
df_complete['fecha_hora_solicitud'] = df_complete['fecha_solicitud'] + (df_complete['hora_solicitud_dt'] - df_complete['hora_solicitud_dt'].dt.normalize())
df_complete

In [ ]:
df_complete['hora_solicitud'] = pd.to_datetime(df_complete['hora_solicitud'], format='mixed', errors='coerce').dt.hour
df_complete

### 4.2.2 Extract day and month

In [ ]:
df_complete['dia_solicitud'] = df_complete['fecha_solicitud'].dt.day_name()
df_complete['mes_solicitud'] = df_complete['fecha_solicitud'].dt.month_name()
df_complete

### 4.2.3 Encode hour

In [ ]:
df_complete['hora_sin'] = np.sin(2 * np.pi * df_complete['hora_solicitud'] / 24)
df_complete['hora_cos'] = np.cos(2 * np.pi * df_complete['hora_solicitud'] / 24)

### 4.2.4 Obtain Target Variable

In [ ]:
df_complete['tiempo_resolución'] = (df_complete['fecha_concluido'] - df_complete['fecha_solicitud']).dt.total_seconds() / 86400
df_complete

In [ ]:
df_complete['tiempo_resolución_con_hora'] = (df_complete['fecha_concluido'] - df_complete['fecha_hora_solicitud']).dt.total_seconds() / 86400
df_complete

In [ ]:
tema_ceros = df_complete[['tema_solicitud','tipo_de_entrada']][df_complete['tiempo_resolución_con_hora'] <= 0]
tema_ceros.value_counts()

In [ ]:
df_complete = df_complete[df_complete['tiempo_resolución_con_hora'] > 0]

## 4.3 Zip Code Cleaning

### 4.3.1 Standarize Zip Code Format

In [ ]:
patterns = df_complete['codigo_postal_solicitud'].dropna().apply(
    lambda x: ''.join(['d' if c.isdigit() else ('a' if c.isalpha() else c) for c in str(x).strip()])
)
patterns.value_counts()

In [ ]:
def clean_cp(x):
    if pd.isna(x):
        return None
    x = str(x).strip().rstrip('.')
    if x == '':
        return None
    try:
        result = str(int(float(x))).zfill(5)
        if len(result) > 5 or int(result) < 1000:
            return None
        return result
    except ValueError:
        return None

In [ ]:
df_complete['codigo_postal_solicitud'] = df_complete['codigo_postal_solicitud'].apply(clean_cp)

In [ ]:
patterns = df_complete['codigo_postal_solicitud'].dropna().apply(
    lambda x: ''.join(['d' if c.isdigit() else ('a' if c.isalpha() else c) for c in str(x).strip()])
)
patterns.value_counts()

## 4.4 Impute Municipality

### 4.4.1 Map Municipality Number to Name

In [ ]:
df_complete['alcaldia_solicitud'].value_counts()

In [ ]:
alcaldías = {2.0:'Azcapotzalco',3.0:'Coyoacán',4.0:'Cuajimalpa de Morelos',5.0:'Gustavo A. Madero',6.0:'Iztacalco',7.0:'Iztapalapa',8.0:'La Magdalena Contreras',
9.0:'Milpa Alta',10.0:'Álvaro Obregón',11.0:'Tláhuac',12.0:'Tlalpan',13.0:'Xochimilco',14.0:'Benito Juárez',15.0:'Cuauhtémoc',16.0:'Miguel Hidalgo',
17.0:'Venustiano Carranza'}

In [ ]:
df_complete['alcaldia_solicitud'] = df_complete['alcaldia_solicitud'].map(alcaldías)
df_complete

In [ ]:
df_complete['alcaldia_solicitud'].value_counts()

### 4.4.2 Impute Municipality with Zip Code

In [ ]:
alcaldia_cp = {
    '01':'Álvaro Obregón',
    '02':'Azcapotzalco',
    '03':'Benito Juárez',
    '04':'Coyoacán',
    '05':'Cuajimalpa de Morelos',
    '06':'Cuauhtémoc',
    '07':'Gustavo A. Madero',
    '08':'Iztacalco',
    '09':'Iztapalapa',
    '10':'La Magdalena Contreras',
    '11':'Miguel Hidalgo',
    '12':'Milpa Alta',
    '13':'Tláhuac',
    '14':'Tlalpan',
    '15':'Venustiano Carranza',
    '16':'Xochimilco'
}

In [ ]:
missing_mask = df_complete['alcaldia_solicitud'].isna()

df_complete.loc[missing_mask, 'alcaldia_solicitud'] = (
    df_complete.loc[missing_mask, 'codigo_postal_solicitud']
    .astype(str)
    .str[:2]
    .map(alcaldia_cp)
)

In [ ]:
df_complete.isnull().mean()

## 4.5 Impute Gender

In [ ]:
df_complete[['sexo']] = df_complete[['sexo']].fillna('No especificado')

In [ ]:
df_complete['sexo'] = df_complete['sexo'].replace(to_replace='3', value='No especificado')

In [ ]:
df_complete['sexo'].value_counts()

## 4.6 Drop Remaining Columns

## 4.7 Drop NAs in Remaining Columns

In [ ]:
df_complete.dropna(subset=['alcaldia_solicitud','codigo_postal_solicitud','tiempo_resolución','tiempo_resolución_con_hora'],inplace=True)
df_complete

In [ ]:
columnas_drop = ['id_folio','fecha_solicitud','hora_solicitud','fecha_hora_solicitud','hora_solicitud_dt','estatus','fecha_concluido']
df_complete.drop(columns=columnas_drop,inplace=True)
df_complete

# 5. Visualizations

In [ ]:
columns_df = df_complete.drop(columns=['tema_solicitud','codigo_postal_solicitud','hora_sin','hora_cos','tiempo_resolución','tiempo_resolución_con_hora']).columns.to_list()
columns_df

## Histogramas

In [ ]:
top_n = 20
conteo_tema = df_complete['tema_solicitud'].value_counts()

conteo_sin_otro = conteo_tema.drop(labels='OTRO')

top_temas = conteo_sin_otro.head(top_n)
resto = conteo_sin_otro.iloc[top_n:].sum() + conteo_tema['OTRO']

serie_plot = pd.concat([top_temas, pd.Series({'OTRO': resto})])

fig = px.bar(x=serie_plot.index, y=serie_plot.values)
fig.update_layout(title='Top 20 temas de solicitud (resto agrupado en "Otro")',xaxis_title='tema_solicitud', yaxis_title='Cantidad')
fig.show()

In [ ]:
for param in columns_df:
    param_values = df_complete[param].value_counts()
    fig = px.histogram(x=param_values.index,y=param_values.values,nbins=len(param_values.index))
    fig.update_layout(title='Cantidad de registros por categoría',xaxis_title=f'{param}', yaxis_title='Cantidad')
    fig.show()

## Boxplots

In [ ]:
top_n = 20
conteo_tema = df_complete['tema_solicitud'].value_counts()
conteo_sin_otro = conteo_tema.drop(labels='OTRO')
top_temas = conteo_sin_otro.head(top_n).index

df_complete['tema_solicitud_agrupado'] = df_complete['tema_solicitud'].where(
    df_complete['tema_solicitud'].isin(top_temas), other='OTRO'
)

y_col = 'tiempo_resolución_con_hora'

fig = px.box(df_complete, x='tema_solicitud_agrupado', y=y_col, points=False)
fig.update_yaxes(type='log', title='tiempo_resolución (escala log)')
fig.update_layout(title='Tiempo de resolución por tema_solicitud (Top 20 + Otro)')
fig.show()

In [ ]:
y_col = 'tiempo_resolución_con_hora'

for param in columns_df:
    fig = px.box(df_complete,x=param,y=y_col, points=False)
    fig.update_yaxes(type='log',title='tiempo_resolución (escala log)')
    fig.update_layout(title=f'Tiempo de resolución por {param}')
    fig.show()

# 6. Correlations

## Numeric

In [ ]:
df_complete['codigo_postal_solicitud'] = df_complete['codigo_postal_solicitud'].astype("float")

In [ ]:
df_complete.corr(numeric_only=True)

In [ ]:
df_complete.drop(columns=['codigo_postal_solicitud'],inplace=True)
df_complete

## Categorical

In [ ]:
lista_categorias = columns_df + ['tema_solicitud_agrupado']
lista_categorias

In [ ]:
dict_resultados = {
    'Categoria' : [],
    'Prueba F' : [],
    'P-value' : [],
    'Eta^2' : []
}
for categoria in lista_categorias:
    lista_variables = df_complete[categoria].value_counts().index.to_list()
    dict_arrays = {}
    for variable in lista_variables:
        array_variable = df_complete['tiempo_resolución_con_hora'][df_complete[categoria] == variable].to_numpy()
        dict_arrays[variable] = array_variable
    F = f_oneway(*dict_arrays.values())
    media_tiempo = df_complete['tiempo_resolución_con_hora'].mean()
    media_por_grupo = df_complete.groupby(categoria)['tiempo_resolución_con_hora'].mean()
    conteo_por_grupo = df_complete.groupby(categoria)['tiempo_resolución_con_hora'].count()
    ss_total = ((df_complete['tiempo_resolución_con_hora'] - media_tiempo)**2).sum()
    ss_efecto = (((media_por_grupo - media_tiempo)**2)*conteo_por_grupo).sum()
    eta_cuadrada = ss_efecto / ss_total
    dict_resultados['Categoria'].append(categoria)
    dict_resultados['Prueba F'].append(F.statistic.round(3))
    dict_resultados['P-value'].append(F.pvalue.round(11))
    dict_resultados['Eta^2'].append(eta_cuadrada)

In [ ]:
df_results = pd.DataFrame(dict_resultados)
df_results

In [ ]:
df_complete.drop(columns='tema_solicitud_agrupado', inplace=True)

# 7. Distributions

In [ ]:
fig = px.histogram(df_complete, x='tiempo_resolución_con_hora', nbins=200)
fig.update_layout(title='Distribución de tiempo de resolución (días)', xaxis_title='Días', yaxis_title='Cantidad')
fig.update_traces(xbins=dict(start=0))
fig.show()

df_complete['tiempo_resolución'].describe()

In [ ]:
tiempo_log = np.log1p(df_complete['tiempo_resolución_con_hora'].to_numpy())
df_tiempo_log = pd.DataFrame({
    'tiempo_resolución_con_hora' : tiempo_log
})
fig = px.histogram(df_tiempo_log, x='tiempo_resolución_con_hora', nbins=200)
fig.update_layout(title='Distribución de tiempo de resolución (días)', xaxis_title='Días', yaxis_title='Cantidad')
fig.update_traces(xbins=dict(start=0))
fig.show()
df_tiempo_log['tiempo_resolución_con_hora'].describe()

# 8. Exportado

In [ ]:
df_complete.to_csv(path_or_buf='../data/preprocessed/dataset_preprocesado.csv', index=False)


In [ ]:
df_check = pd.read_csv('../data/preprocessed/dataset_preprocesado.csv')
df_check.head()